In [4]:
!pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 84.4 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.0
    Uninstalling pip-26.0:
      Successfully uninstalled pip-26.0


In [5]:
!pip -q install transformers datasets scikit-learn torch pandas numpy

In [6]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertModel
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.svm import LinearSVC
import scipy.linalg

In [7]:
class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }

In [8]:
def get_bert_embeddings(model, data_loader, device):
    model = model.eval()
    embeddings = []
    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            hidden_states = outputs.last_hidden_state
            cls_embeddings = hidden_states[:, 0, :]
            embeddings.append(cls_embeddings.cpu().numpy())
    return np.vstack(embeddings)

def get_rowspace_projection(W):
    if W.ndim == 1:
        W = W.reshape(1, -1)
    
    basis = scipy.linalg.orth(W.T)
    P_row = basis @ basis.T
    return P_row

def inlp(X, Z, n_iterations):
    X_projected = X.copy()
    P_final = np.eye(X.shape[1])
    
    for i in range(n_iterations):
        clf = LinearSVC(dual='auto', max_iter=2000)
        clf.fit(X_projected, Z)
        W = clf.coef_
        
        P_row = get_rowspace_projection(W)
        P_null = np.eye(X.shape[1]) - P_row
        
        P_final = P_null @ P_final
        X_projected = X_projected @ P_null.T
        
    return P_final, X_projected

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased').to(device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 762.42it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
df = pd.read_csv('train.csv')

df['y_label'] = (df['target'] >= 0.5).astype(int)

df = df.loc[(df['male'] >= 0.5) | (df['female'] >= 0.5)].copy()
df['z_label'] = (df['female'] >= 0.5).astype(int)

df_sample = df.sample(n=5000, random_state=42)

texts = df_sample['comment_text'].values
y = df_sample['y_label'].values
z = df_sample['z_label'].values

dataset = TextDataset(texts, tokenizer)
data_loader = DataLoader(dataset, batch_size=32, num_workers=2)

X = get_bert_embeddings(bert_model, data_loader, device)

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [ ]:
X_train, X_test, y_train, y_test, z_train, z_test = train_test_split(
    X, y, z, test_size=0.3, random_state=42
)

In [ ]:
main_clf = LogisticRegression(max_iter=1000)
main_clf.fit(X_train, y_train)
y_pred_orig = main_clf.predict(X_test)
print(f"Original Accuracy (Task Y): {accuracy_score(y_test, y_pred_orig):.4f}")

In [ ]:
gender_clf = LogisticRegression(max_iter=1000)
gender_clf.fit(X_train, z_train)
z_pred_orig = gender_clf.predict(X_test)
print(f"Original Accuracy (Protected Z): {accuracy_score(z_test, z_pred_orig):.4f}")

In [ ]:
P, X_train_inlp = inlp(X_train, z_train, n_iterations=30)
X_test_inlp = X_test @ P.T

main_clf_inlp = LogisticRegression(max_iter=1000)
main_clf_inlp.fit(X_train_inlp, y_train)
y_pred_inlp = main_clf_inlp.predict(X_test_inlp)
print(f"INLP Accuracy (Task Y): {accuracy_score(y_test, y_pred_inlp):.4f}")

In [ ]:
gender_clf_inlp = LogisticRegression(max_iter=1000)
gender_clf_inlp.fit(X_train_inlp, z_train)
z_pred_inlp = gender_clf_inlp.predict(X_test_inlp)
print(f"INLP Accuracy (Protected Z): {accuracy_score(z_test, z_pred_inlp):.4f}")